In [1]:
import sqlalchemy
import sys
import os

from dotenv import load_dotenv

src_path = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "src"))
sys.path.append(src_path)

from libs import sql_connector
import pandas as pd

In [2]:
load_dotenv(override=True)
DB_USER = os.getenv("RAD_USER")
DB_PASSWORD = os.getenv("RAD_PASSWORD")
DB_SERVER = os.getenv("RAD_SERVER")
DB_DATABASE = os.getenv("RAD_DATABASE")

sql_connector = sql_connector.SQLConnector(
            DB_USER,
            DB_PASSWORD,
            DB_SERVER,
            DB_DATABASE,
        )

In [3]:
QUERY_FILE = "../../sql/nonprod_rad.sql"

rad_df = sql_connector.query_from_file(QUERY_FILE)

In [4]:
pi_form_df = pd.read_csv("pi_form.csv", encoding="ISO-8859-1", skiprows=1)

In [5]:
# Define regex with two capturing groups (AWD- first for precedence)
pattern = r"(AWD-\d{6})|(GR\d{6})"

# Extract the matched pattern
pi_form_df["identifier"] = pi_form_df["UW Award #"].str.extract(pattern).bfill(axis=1).iloc[:, 0]

print(pi_form_df["identifier"])

0         GR000015
1              NaN
2              NaN
3              NaN
4       AWD-000000
           ...    
4162    AWD-007335
4163    AWD-004032
4164      GR007429
4165      GR013108
4166      GR042485
Name: identifier, Length: 4167, dtype: object


In [6]:
merged_df = rad_df.merge(pi_form_df, left_on="WorkdayAwardNumber", right_on="identifier", how="inner")

In [7]:
filtered_df = merged_df[
    (~merged_df["isAnimalUse"].isna()) &
    (~merged_df["isHumanSubjects"].isna()) & 
    (merged_df["mod_status"] == "Processed") &
    (~merged_df["modifiedSponsorAwardedTotal"].isna()) &
    (~merged_df["Is this a temporary internal extension request?\n"].isna()) &
    (merged_df["AuthorizedAmount"] > 0) &
    (merged_df["BilledToDateAmount"] > 0) &
    (~merged_df["Is this a 2nd NIH No-Cost Extension request?"].isna()) &
    (~merged_df["projectType"].isna()) &
    (~merged_df["pi_name"].isna()) &
    (~merged_df["Are there NEW cost share commitments during the extension that have not previously been documented on a cost share addendum?\n\nIf cost share was committed in the original proposal and documented on..."].isna())
    ]

### Plausible modification requests for demo (processed):

In [8]:
filtered_df[["displayIdentifier", "identifier", "Completion time"]].rename(columns={
    "displayIdentifier": "mod_id",
    "identifier": "award_id",
    "Completion time": "completion_time"
})

,mod_id,award_id,completion_time
2,MOD40645,AWD-000367,8/15/24 12:43:48
12,MOD31459,AWD-000391,12/4/23 13:29:11
14,MOD34503,AWD-000443,8/25/23 16:47:43
17,MOD26449,AWD-000289,6/20/24 15:53:09
20,MOD39097,AWD-000289,6/20/24 15:53:09
...,...,...,...
8525,MOD34305,AWD-000291,12/28/23 14:22:01
8526,MOD34305,AWD-000291,9/20/24 8:54:01
8527,MOD34305,AWD-000291,12/17/24 10:06:52
8528,MOD28557,AWD-000315,5/31/24 12:07:40


### Plausible modification requests for demo (in progress):

In [10]:
filtered_df2 = merged_df[
    (~merged_df["isAnimalUse"].isna()) &
    (~merged_df["isHumanSubjects"].isna()) & 
    (merged_df["mod_status"] == "OSP Assigned") &
    (~merged_df["modifiedSponsorAwardedTotal"].isna()) &
    (~merged_df["Is this a temporary internal extension request?\n"].isna()) &
    (merged_df["AuthorizedAmount"] > 0) &
    (merged_df["BilledToDateAmount"] > 0) &
    (~merged_df["Is this a 2nd NIH No-Cost Extension request?"].isna()) &
    (~merged_df["projectType"].isna()) &
    (~merged_df["pi_name"].isna()) &
    (~merged_df["Are there NEW cost share commitments during the extension that have not previously been documented on a cost share addendum?\n\nIf cost share was committed in the original proposal and documented on..."].isna())
    ]

In [15]:
filtered_df2[["displayIdentifier", "identifier", "Completion time"]].rename(columns={
    "displayIdentifier": "mod_id",
    "identifier": "award_id",
    "Completion time": "completion_time"
}).head(10)

,mod_id,award_id,completion_time
393,MOD44887,AWD-001420,11/19/23 13:11:41
394,MOD44887,AWD-001420,12/13/24 11:07:59
1315,MOD45852,AWD-002396,5/31/24 10:10:46
1316,MOD45861,AWD-002396,5/31/24 10:10:46
1357,MOD40303,AWD-002475,10/17/23 10:20:19
1676,MOD44239,AWD-002851,9/1/23 13:15:44
2856,MOD45856,AWD-004123,1/25/25 9:47:37
2968,MOD45270,AWD-003171,1/29/24 13:35:34
2969,MOD45270,AWD-003171,12/30/24 13:10:15
3717,MOD45921,AWD-004982,4/3/24 12:19:51


It is strange that some records in this set had PI forms completed a long time ago but still have review status `OSP Assigned`. Filter these below.

In [ ]:
import datetime

datetime.